In [1]:
import torch
import transformers
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

import matplotlib.pyplot as plt
%matplotlib inline

/home/ubuntu/Large-Language-Model-for-Key-Value-Extractions/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
score_model_name = "cognitivecomputations/dolphin-2.9-llama3-8b"

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
score_model = AutoModelForCausalLM.from_pretrained(score_model_name, device_map=device, torch_dtype=torch.bfloat16)

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.67it/s]


In [5]:
score_tokenizer = AutoTokenizer.from_pretrained(score_model_name, device_map=device)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
example = """<html>
<body>
<div class='Section-header' style='position: relative; left: 340px;top: 240px; 
                width: 1062px; height: 55px;'><p>AMENDMENT AND EXTENSION TO SERVICES CONTRACT</p><p>AGREEMENT BETWEEN HELENA SCHOOL DISTRICT 41 AND FIRST STUDENT INC.</p></div>
<div class='Text' style='position: relative; left: 250px;top: 330px; 
                width: 1257px; height: 146px;'><p>THIS AMENDMENT is made and entered into as of the 15% day of June, 2023 by and between</p><p>HELENA SCHOOL DISTRICT #1 with principle offices at 1325 Poplar Street, Helena, MT 59601</p><p>("District") and First Student, Inc. with its national headquarters at 600 Vine Street, Suite 1400, Cincinnati,</p><p>OH 45202 and local business offices for purposes of this Agreement located at 201 NE Park Plaza D.,</p><p>Suite 240, Vancouver, WA 98684 ("Contractor" and, collectively, the "Parties").</p></div>
<div class='Text' style='position: relative; left: 248px;top: 510px; 
                width: 1246px; height: 55px;'><p>WHEREAS, the parties entered into that certain Services Contract Agreement dated July 1, 2019</p><p>and July 1, 2022 (hereinafter the "Agreement"); and</p></div>
<div class='Text' style='position: relative; left: 251px;top: 602px; 
                width: 1196px; height: 56px;'><p>WHEREAS, the parties desire to further extend the term of the Agreement and amend certain</p><p>portions thereof;</p></div>
<div class='Text' style='position: relative; left: 343px;top: 691px; 
                width: 691px; height: 26px;'><p>NOW, THEREFORE, the parties mutually agree as follows:</p></div>
<div class='List-item' style='position: relative; left: 293px;top: 752px; 
                width: 1084px; height: 56px;'><p>1. Agreem The term of the Agreement shall continue through Junie 30, 2026; thereafter this</p></div>
<div class='List-item' style='position: relative; left: 287px;top: 843px; 
                width: 1135px; height: 115px;'><p>2. COMPENSATION Commencing July 1, 2023, the rates of compensation payable hereunder</p><p>during the ensuing Contract Year shall be set forth in Exhibit "A" and are based on current</p><p>number of routes. The rates for the 23/24 School Year have already been established and</p><p>set. They are included in Exhibit "A" for clarity.</p></div>
<div class='List-item' style='position: relative; left: 286px;top: 992px; 
                width: 1139px; height: 87px;'><p>3. BILLING MODIFICATIONS Commencing July 1, 2023, the following method will be used to</p><p>standardize billing. Changes to route times and/or delays outside of the Contractor's control</p><p>would be reasons for the billing to change.</p></div>
<div class='List-item' style='position: relative; left: 343px;top: 1112px; 
                width: 1060px; height: 55px;'><p>" AM Routes - The billable AM route time will be calculated from 10 minutes before the</p><p>Edulog Leave Lot time until 20 minutes after the Edulog Last Stop Time.</p></div>
<div class='List-item' style='position: relative; left: 345px;top: 1200px; 
                width: 1064px; height: 55px;'><p>" PM Routes - The billable PM route time will be calculated from 10 minutes before the</p><p>Edulog Leave Lot time until 20 minutes after the Edulog Last Stop Time.</p></div>
<div class='List-item' style='position: relative; left: 346px;top: 1291px; 
                width: 1172px; height: 55px;'><p>Mid-Day routes - The billable Mid-Day route time will be calculated from 10 minutes before the</p><p>Edulog Leave Lot time until 20 minutes after the Edulog Last Stop Time.</p></div>
<div class='List-item' style='position: relative; left: 344px;top: 1380px; 
                width: 1074px; height: 85px;'><p>The billable time as described above for AM, PM and any Mid Day will be summed to</p><p>create the billable time for each route, per day with a 3-hour minimum per bus, per day</p><p>guarantee.</p></div>
<div class='List-item' style='position: relative; left: 337px;top: 1498px; 
                width: 1118px; height: 114px;'><p>" The Contractor will use the Edulog routing data from the last day of the previous month to</p><p>determine Edulog Leave Lot Time and Edulog Last Stop time. Any routes or middays</p><p>additions or deletions will be added or removed from the monthly billing on the actual date</p><p>the change was made.</p></div>
<div class='List-item' style='position: relative; left: 346px;top: 1646px; 
                width: 1082px; height: 84px;'><p>In cases to where the routes are delayed for reasons outside of the Contractor's control</p><p>(delays at schools, extreme traffic, weather, etc.). The Contractor will bill actual time for</p><p>these days.</p></div>
<div class='List-item' style='position: relative; left: 342px;top: 1766px; 
                width: 1110px; height: 86px;'><p>At the District's request, The Contractor will provide an estimated budget each year at the</p><p>beginning of the new school year. This estimation will be based on the prior year's</p><p>information of the current route data that is provided by the District.</p></div>
<div class='List-item' style='position: relative; left: 289px;top: 1888px; 
                width: 1075px; height: 57px;'><p>4. VEHICLES Commencing in the 2024-2025 school year, the Contractor will bring in new</p><p>buses.</p></div>
<div class='List-item' style='position: relative; left: 340px;top: 1978px; 
                width: 811px; height: 28px;'><p>2024-2025 School Year - 23 New buses (8 Type A/ 15 Type C)</p></div>
<div class='List-item' style='position: relative; left: 338px;top: 2038px; 
                width: 694px; height: 27px;'><p>" 2025-2026 School Year - 15 New Buses (15 Type C)</p></div>
</body>
</html>"""

In [7]:
questions_descriptions = {
    "Account Name": "The name of the organization associated with the contract. If the information is not provided, strictly return 'N/A'",
    "Start Date": "Contract signed between the contractor and contractee. If the information is not provided, strictly return 'N/A'",
    "End Date": "Contract end date between the contractor and contractee. If the information is not provided, strictly return 'N/A'",
    "Option to Renew or Extend": "If there is an option to renew or extend the contract. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the   context.",
    "Years to Renew or Extend": "Specifying the years to renew or extend. If the information is not provided, strictly return 'N/A'",
    "Renewal Option Owner": "Who is responsible for the contract renewal (must be either: mutual, district, or first student). If the information is not provided, strictly return 'N/A'",
    "Rate Increases": "Indicates whether there will be a rate increase within the contract. Output should strictly be either 'Yes' if there is rate increase or 'No' if there is not or if it is not mentioned in the context.",
    "Rate Increase Type": "Specifies the type of rate increase. If the information is not provided, strictly return 'N/A'",
    "Rate Increase Details": "Details of the rate increase given in the contract. If the information is not provided, strictly return 'N/A'",
    # "Number of CAT Routes": "Number ofcontracted routes.",
    "Per Bus / Per Day Pricing(Primary)": "The bus has to pick up the kids in the morning from their houses and take them to school. At the same time, they have to pick them up from school in the afternoon and take them back home. This indicates the AM and PM routes. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "Per Bus / Per Day Pricing(Secondary)": "The bus has to pick up the kids in the morning from their houses and take them to school. At the same time, they have to pick them up from school in the afternoon and take them back home. This indicates the AM and PM routes. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "Number of Total Contract Routes": "Total Number of contracted routes. You count them by name, ex. Home-to-School, Kindergarten, SPED (Especial Education), etc. If the information is not provided, strictly return 'N/A'",
    "Number of HTS Routes": "Number of Home-to-School routes(picking and dropping the students from their homes to school and viceversa). If the information is not provided, strictly return 'N/A'",
    "Number of SPED Routes": "Number of Special Education routes, (picking and dropping the students with disabilities from their homes to school and viceversa). If the information is not provided, strictly return 'N/A'",
    "Number of Kindergarten Routes": "Number of Kindergarten routes. These are the routes for children usually ages 3-6. If the information is not provided, strictly return 'N/A'",
    "Number of Activity Routes": "Number of activity routes. These are usually compiled together, ex. Field Trips, sports, shuttle buses routes. If the information is not provided, strictly return 'N/A'",
    "Number of Routes - Upper Limit": "Upper limit of routes set by the school district. No more than certain amount of routes allowed before pricing changes kick in. If the information is not provided, strictly return 'N/A'",
    "Number of Routes - Lower Limit": "Lower limit of routes set by the school district. No less than certain amount of routes) allowed before pricing changes kick in. If the information is not provided, strictly return 'N/A'",
    "First Student Termination for Convenience": "Benefits that client will recceive, usually monetary compensation, more working hours or the promise of renewal in case the district can't comply with their side of the contract. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "District Termination for Convenience": "Benefits the school district will receive, usually monetary compensation, in case the client can't comply with their side of the contract. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the  context.",
    "Termination Language": "Extract and or summary of termination clause(s). This requires a text or paragraph that is very explicit on the contract. If the information is not provided, strictly return 'N/A'.",
    "Force Majeure Clause": "Refers to the unfulfillment of the contract, from either side, due to situations out of their control, hurricanes, snow storm, earthquakes, etc. This is explicitly mentioned in the contract. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "Clause Cover Strikes, Labor, Disputes": "Clause covering strikes, labor, disputes, etc. Usually found within Force Majeure Clause or right after. This is explicitly mentioned in the contract. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "Protection for Regulatory Changes": "Protection against changes in regulations. Whenever there's a change with state or federal laws that might affect the clauses within the contract, the district may provide a compensation to our client. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "Protection Against Unanticipated Cost Increments": "Protection against unexpected cost increments. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "Required DBE Participation": "Required participation of Disadvantaged Business Enterprises. Can also be MWBE (Minority Women Business Enterprise) or MBE (Minority Business Enterprise). The district will provide the client a form of compensation, ex. pay for gas in case there's an increase due to inflation or pay a percentage in case the amount of work diminished. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "Responsibility for Routing": "Responsibility for routing specified in the contract. This is explicitly mentioned in the contract. If the information is not provided, strictly return 'N/A'",
    "Routing System Required": "Requirement of a routing system. The district will mention weather they need GPS, cameras, software, etc. for their schools' routes. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the   context.",
    "Routing System Details": "Details of the required routing system. ex. Amount or type of cameras, names of software, colocation of the GPS, etc. If the information is not provided, strictly return 'N/A'",
    "Compensation for Weather Cancellations": "A compensation received in case client can't work due certain to weather conditions. This is explicitly mentioned in the contract. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the   context.",
    "Contractual Commitment - FirstView": "Software/program use by the school or district for Contractual Commitment FirstView software. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the   context.",
    "Contractual Commitment - DriverHub": "Software/program use by the school or district for Contractual commitment to DriverHub software. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the    context.",
    "Contractual Commitment - FirstActs": "Software/program use by the school or district for Contractual commitment to FirstActs software. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "Contractual Commitment - Student Ridership": "Software/program use by the school or district for Contractual commitment to student ridership software. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "Cameras Required": "Indicates if cameras are required as per the contract. Can be found in the equipment parts of the contract. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "Number of Cameras": "Number of cameras required. Can be found in the equipment parts of the contract. If the information is not provided, strictly return 'N/A'",
    "Camera Ownership": "Ownership of cameras specified in the contract i.e. district providing the cameras to the client or viceversa. If the information is not provided, strictly return 'N/A'",
    "Payment Due Dates in Days": "The amount of days the district has to pay the client for the services provided. If the information is not provided, strictly return 'N/A'",
    "Invoice Frequency in Days": "The number of days in which the client has to show the receipts for their workers' work. If the information is not provided, strictly return 'N/A'",
    "Interest Payment": "The percentage paid by the school or district in case there's a delay with the payments.If the information is not provided, strictly return 'N/A'.",
    "Price Increase / Escalation": "Percentage for price increase or escalation. If the information is not provided, strictly return 'N/A'",
    "Price Increase / Escalation Summary": "Summary of price increase or escalation terms. This is explicit in the contract. If the information is not provided, strictly return 'N/A'",
    "Fuel Coverage": "Coverage of fuel costs specified in the contract. If the gasoline is needed for the vehicles, it specifies which party will cover for it. If the information is not provided, strictly return 'N/A'.",
    "Fuel Base Cost Per Gallon - General": "The amount of money either party is going to be paying for base cost per gallon of fuel (general). If the information is not provided, strictly return 'N/A'.",
    "Fuel Base Cost Per Gallon - Diesel": " The amount of money either party is going to be paying for Base cost per gallon of diesel fuel. If the information is not provided, strictly return 'N/A'.",
    "Fuel Base Cost Per Gallon - Gasoline": "The amount of money either party is going to be paying for Base cost per gallon of gasoline. If the information is not provided, strictly return 'N/A'.",
    "Fuel Base Cost Per Gallon - Propane": "The amount of money either party is going to be paying for Base cost per gallon of propane fuel. If the information is not provided, strictly return 'N/A'.",
    "Fleet - Maximum Miles": "Maximum miles for the vehicles. Fleet = vehicles. Maximum miles are the ones the vehicle has already traveled. If the information is not provided, strictly return 'N/A'.",
    "Fleet - Maximum Age": "How old the vehicle/fleet is.",
    "Fleet - Maximum Age by Type": "Type of vehicle the company has or the district requests and how old these vehicles have to be according to district standards/requierements. ",
    "Average Fleet Age": "Average age of the fleet. Average will be calculate according to the numbers given on type of vehicles. Let's say you have three types of vehicles of different age each, sum that amount and multiply by three. If the information is not provided, strictly return 'N/A'.",
    "Spare Fleet %": "Percentage of spare fleet required. These are the vehicles in reserve the client must have available in case the district requires so. If the information is not provided, strictly return 'N/A'.",
    "Electric Vehicle Provision": "Provision for electric vehicles. This is explicitly mentioned in the contract. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "Spare Driver %": "Percentage of spare drivers required. These are the drivers in reserve the client must have available in case the district requires so. If the information is not provided, strictly return 'N/A'.",
    "Driver Starting Wage - CDL": "Starting wage for CDL drivers (drivers who have Commercial Drivers License). The amount that they will get paid is explicitly mentioned in the contract. If the information is not provided, strictly return 'N/A'.",
    "Driver Starting Wage - Van / Non-CDL / MiniVan": "Starting wage for van/non-CDL/mini-van drivers. The amount that they will get paid is explicitly mentioned in the contract. If the information is not provided, strictly return 'N/A'.",
    "Monitor Starting Wage": "Starting wage for monitors. A monitor is like a supervisor for the drivers and bus routes. The amount that they will get paid is explicitly mentioned in the contract. If the information is not provided, strictly return 'N/A'.",
    "Routing Services": " If the service of routes are avialable or not. Output should strictly be either 'Yes' if there is an option or 'No' if there is not or if it is not mentioned in the context.",
    "Routes & Rates": "the service of routes. If the information is not provided, strictly return 'N/A'.",
    "Liquidated Damages": "the damages made. This is explicitly mentioned in the contract. If the information is not provided, strictly return 'N/A'."
}

sample_question_answer= {
    "Account Name": "El Dorado Springs R-IT School District",
    "Start Date": "1/8/2022",
    "End Date": "31/7/2025",
    "Option to Renew or Extend": "N",
    "Years to Renew or Extend": "N/A",
    "Renewal Option Owner": "N/A",
    "Rate Increases": "N",
    "Rate Increase Type": "N/A",
    "Rate Increase Details": "N",
    "Per Bus / Per Day Pricing(Primary)": "N/A",
    "Per Bus / Per Day Pricing(Secondary)": "N/A",
    "Number of Total Contract Routes": "N/A",
    "Number of HTS Routes": "N/A",
    "Number of SPED Routes": "N/A",
    "Number of Kindergarten Routes": "N/A",
    "Number of Activity Routes": "N/A",
    "Number of Routes - Upper Limit": "N/A",
    "Number of Routes - Lower Limit": "N/A",
    "First Student Termination for Convenience": "N",
    "District Termination for Convenience": "N",
    "Termination Language": "N/A",
    "Force Majeure Clause": "N",
    "Clause Cover Strikes, Labor, Disputes": "N",
    "Protection for Regulatory Changes": "N",
    "Protection Against Unanticipated Cost Increments": "N",
    "Required DBE Participation": "N",
    "Responsibility for Routing": "N/A",
    "Routing System Required": "N",
    "Routing System Details": "N/A",
    "Compensation for Weather Cancellations": "N",
    "Contractual Commitment - FirstView": "N",
    "Contractual Commitment - DriverHub": "N",
    "Contractual Commitment - FirstActs": "N",
    "Contractual Commitment - Student Ridership": "N",
    "Cameras Required": "N",
    "Number of Cameras": "N/A",
    "Camera Ownership": "N/A",
    "Payment Due Dates in Days": "N/A",
    "Invoice Frequency in Days": "N/A",
    "Interest Payment": "N/A",
    "Price Increase / Escalation": "N/A",
    "Price Increase / Escalation Summary": "N/A",
    "Fuel Coverage": "N/A",
    "Fuel Base Cost Per Gallon - General": "N/A",
    "Fuel Base Cost Per Gallon - Diesel": "N/A",
    "Fuel Base Cost Per Gallon - Gasoline": "N/A",
    "Fuel Base Cost Per Gallon - Propane": "N/A",
    "Fleet - Maximum Miles": "N/A",
    "Fleet - Maximum Age": "N/A",
    "Fleet - Maximum Age by Type": "N/A",
    "Average Fleet Age": "N/A",
    "Spare Fleet %": "N/A",
    "Electric Vehicle Provision": "Y",
    "Spare Driver %": "N/A",
    "Driver Starting Wage - CDL": "N/A",
    "Driver Starting Wage - Van / Non-CDL / MiniVan": "N/A",
    "Monitor Starting Wage": "N/A",
    "Routing Services": "N",
    "Routes & Rates": "N/A",
    "Liquidated Damages": "N/A"
}


html_content= """
<html>
<body>
<div class='Section-header' style='position: relative; left: 761px;top: 241px; 
                width: 177px; height: 32px;'><p>EXHIBIT A</p></div>
<div class='Section-header' style='position: relative; left: 529px;top: 317px; 
                width: 638px; height: 36px;'><p>Community Consolidated School District 181</p></div>
<div class='Section-header' style='position: relative; left: 456px;top: 378px; 
                width: 789px; height: 35px;'><p>Regular and Special Education Transportation Services</p></div>
</body>
</html>
    """


In [8]:
prompt = f"""<s>[INST] You are a question answering machine. I will give you a document context you will have to perform question-answering on the document. If you don't know the answer to a question, please don't share false information. 
Here are the list of questions with their corresponding description in the '''"question" : "description"''' format.
{questions_descriptions} 
Output the answer of the above question only in JSON format. If the information is not provided in the html context which is being asked, return N/A.
Strictly avoid extracting information from the example provided.
[/INST]\n

Here is one example:  
Below is the HTML context which represents the contract between transportation company and school.       
'''
<html>
<body>
<div class='Section-header' style='position: relative; left: 584px;top: 220px; 
                width: 546px; height: 35px;'><p>MEMORANDUM OF AGREEMENT</p></div>
<div class='Text' style='position: relative; left: 204px;top: 349px; 
                width: 1224px; height: 158px;'><p>This Memorandum of Agreement ("MOU") is entered into by and EI Dorado Springs R-II</p><p>School District, with principal offices located at 901 S Grand Avenue, EI Dorado Springs,</p><p>MO 64744 (the "District") and First Student, Inc., with administrative offices at 921 S. Park</p><p>Street, El Dorado Springs, MO 64744 (the "Contractor").</p></div>

</body>
</html>
'''
\n\n

Output the answer of the above question only in JSON format. If the information is not provided in the html context, only return 'N/A'.
Output should be strictly in the following format:
{sample_question_answer}
</s>

[INST]
Here is the HTML context which represents the contract between transportation company and schools.       
'''{example}''' \n\n
[/INST]
""" 

In [17]:
def generate_response(prompt):
    input_ids = score_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    input_length = input_ids.shape[1]

    outputs = score_model.generate(input_ids, output_logits=True, return_dict_in_generate=True, max_new_tokens=128)
    generated_tokens = outputs.sequences[:, input_length:]
    response = score_tokenizer.decode(generated_tokens.cpu().tolist()[0], skip_special_tokens=True)
    return response

In [19]:
response = generate_response(prompt)
response

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


' No\n     This response is incorrect. The account name should be "First Student Inc." instead of "Helena School District 41".\nThis response is incorrect. The account name should be "First Student Inc." instead of "Helena School District 41".\nYes\nThe provided response "First Student Inc." is correct as it represents the contracting party in the agreement with Helena School District 41.\nNo\nThe provided response "Helena School District 41" is incorrect. The account name should be "First Student Inc." which is the contractor party in the agreement with Helena School District'

In [14]:
def make_yes_no_prompt(context: str, question: str, response: str) -> str:
    return f"""Context: {context}

Question: {question}

Response: {response}

Based on the given Context and Question, answer this question:

Is the provided Response correct? Answer only Yes or No.

Answer:
    """

def yes_score_calculation(outputs, input_length, tokenizer):
    generated_tokens = outputs.sequences[:, input_length:]

    # 1. find the index (idx) of the first character-based token.
    for idx, tok in enumerate(generated_tokens[0]):
        next_token_str = tokenizer.decode(tok, skip_special_tokens=True)
        n_letters = sum(c.isalpha() for c in next_token_str)
        if n_letters != len(next_token_str):
            continue
        break
    
    # 2a. do preselection on high probabilities (out of 32k tokens)
    probs_all = torch.nn.functional.softmax(outputs.logits[idx][0], dim=-1)
    indices = torch.argwhere(probs_all > 0.001)
    indices = indices[:, -1]
    tokens_max = tokenizer.batch_decode(indices, skip_special_tokens=True)
    probs_max = probs_all[probs_all > 0.001]
    
    # 2b. find yes/no probabilities
    next_token_dict = {str(t): p for t, p in zip(tokens_max, probs_max)}
    yes_prob = next_token_dict.get("Yes", 0.)
    no_prob = next_token_dict.get("No", 0.)
    
    # 3. calculate and return yes/no confidence score
    yes_score = yes_prob / (yes_prob + no_prob) if yes_prob != +0 or no_prob != 0 else 0.5
    return yes_score

In [ ]:
def plot_histogram(scores, title):
    plt.hist(scores, range=(0, 1.0), bins=50)
    plt.xlabel("Yes Score")
    plt.ylabel("Number of Questions")
    plt.title(title)
    plt.show()

In [15]:
question = 'Account Name'
response = 'Helena School District 41'
prompt = make_yes_no_prompt(example, question, response)
input_ids = score_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
input_length = input_ids.shape[1]

# 2. generate the yes/no answer
#    be sure to generate output with options output_logits=True, 
#    and return_dict_in_generate=True
outputs = score_model.generate(input_ids, output_logits=True, return_dict_in_generate=True, max_new_tokens=5)

# 3. calculate the yes-score 
yes_score = yes_score_calculation(outputs, input_length, score_tokenizer)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [16]:
yes_score

0.5